# Activity 3 — Nighttime Lights, and `config.py`

Two things happen in this notebook.

**The satellite.** NASA's Black Marble product measures artificial light at night at
500 m resolution, every night, since 2012, everywhere on Earth. It is a proxy for
economic activity that arrives with a lag of about a week with as much granularity as desired.


**The configuration file.** This is the first activity that reads `config.py`.
Up to now every notebook was self-contained: a hardcoded dataset or output filename.
The config file centralizes edits into one file: it becomes clear what we can easily edit.

The rule is:

> **You edit `config.py`. You do not edit the notebook.**

By the end of this activity you will point the same unmodified code at your own
country and a date range you choose, and it will run.

---

## Before you start: the `geo` environment

This notebook needs GDAL, HDF5 and geospatial libraries that do not live in the
default environment. Run it with the **`geo`** kernel.

## The config loader

This cell is the same in every notebook that uses `config.py`. It:

1. finds `config.py`;
2. loads it as a module called `cfg`, so every setting is available as `cfg.COUNTRY`,
   `cfg.TILES`, and so on;
3. derives the standard folder paths into `P`;
4. imports the heavy lifting from `ntl_helpers.py`, which keeps this notebook thin.

Read the banner it prints. Everything in it came from `config.py`.

In [ ]:
# ===========================================================================
# CONFIG LOADER  —  edit config.py, NOT this cell.
# Gives you `cfg` (every country knob) and `P` (the standard paths).
# ===========================================================================
import os, sys, importlib, importlib.util

# config.py and ntl_helpers.py live in activity/, beside this notebook.
# From activity/ that is simply the current folder; from solutions/ it is next door.
CODE_DIR = os.getcwd()
if not os.path.exists(os.path.join(CODE_DIR, "config.py")):
    CODE_DIR = os.path.abspath(os.path.join(CODE_DIR, "..", "activity"))

_spec = importlib.util.spec_from_file_location("config", os.path.join(CODE_DIR, "config.py"))
cfg = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(cfg)
P = cfg.derive_paths(CODE_DIR)

if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
import ntl_helpers
importlib.reload(ntl_helpers)      # pick up edits without restarting the kernel
from ntl_helpers import *

for _k in ("raw", "data", "logs", "ntl_tmp"):
    os.makedirs(P[_k], exist_ok=True)

print(f"[CONFIG] {cfg.COUNTRY} ({cfg.CODE}/{cfg.GADM_ISO3})")
print(f"         tiles      : {cfg.TILES}   (download: {cfg.download_tiles()})")
print(f"         window     : {cfg.NTL_DATE_RANGE or 'ALL available dates'}")
print(f"         outputs    : raw/{cfg.NTL_SHAPE_CSV}, raw/{cfg.NTL_OUTPUT_CSV}")
print(f"         tile store : {P['nl_root']}")

## The config file

Open `config.py` — it is in this same code folder.

Two key settings drive everything below:

```python
COUNTRY        = "The Bahamas"
NTL_DATE_RANGE = "2026-01-05 2026-01-18"
```

`COUNTRY` must be a key of the `COUNTRIES` registry in that file.

The cell below prints the registry so you can see what is on offer, and what each
country would resolve to.

In [ ]:
import pandas as pd

registry = pd.DataFrame([
    {"country": name,
     "code":    v["iso2"],
     "gadm":    v["iso3"],
     "tiles":   " ".join(v["tiles"]),
     "n_tiles": len(v["tiles"])}
    for name, v in cfg.COUNTRIES.items()
]).set_index("country").sort_index()

print("Currently selected:", cfg.COUNTRY, "\n")
registry

## Your NASA token

The satellite archive is free but not anonymous. You need an Earthdata account and a
token of your own:

1. Register at <https://urs.earthdata.nasa.gov>
2. Sign in at <https://ladsweb.modaps.eosdis.nasa.gov> → **My Account → Generate Token**
3. Paste the string into `NASA_TOKEN` in `config.py`

Tokens expire, typically after 60 days. When downloads suddenly start returning `401`,
that is almost always the reason — generate a new one.

The next cell does not download anything if your token is empty. It tells you what to
do and carries on, so the rest of the notebook still runs on whatever tiles are
already on disk.

In [ ]:
api_key   = cfg.NASA_TOKEN
HAVE_TOKEN = bool(api_key) and not api_key.startswith("PASTE")

# What is already on disk, tile by tile?
inventory = {t: len(list_tile_rasters(P["nl_root"], [t], make=True)) for t in cfg.TILES}
print("Tiles on disk:")
for t, n in inventory.items():
    print(f"   {t}: {n:,} file(s)")

if not HAVE_TOKEN:
    print("\nNASA_TOKEN is empty in config.py -> skipping download.")
    print("Set it to run the download cell below; the rest of the notebook will")
    print("still work on the files already present.")

## Downloading

`missing_targets` asks the LAADS server which days exist for your tiles and window,
compares that against what is on disk, and returns only the difference.

`download_many` fetches files in parallel, with retries, writing to `.part` files
and renaming on success so an interrupted run never leaves a half-written tile behind.

Both in `ntl_helpers.py`. The notebook calls these programs.

In [ ]:
from datetime import datetime

# Resolve the window from config.py
if cfg.NTL_DATE_RANGE:
    _s, _e = cfg.NTL_DATE_RANGE.split()
    start_date, end_date = datetime.strptime(_s, "%Y-%m-%d"), datetime.strptime(_e, "%Y-%m-%d")
    print(f"Window: {start_date.date()} -> {end_date.date()}")
else:
    start_date = end_date = None
    print("Window: the entire archive (2012 to today). This takes hours -- set "
          "NTL_DATE_RANGE in config.py before running it for real.")

log_path = os.path.join(P["logs"], "ntl_download_log.txt")

if HAVE_TOKEN:
    target = missing_targets(cfg.download_tiles(), api_key, P["nl_root"],
                             start_date=start_date, end_date=end_date)
    print(f"{len(target)} file(s) available on the server but missing on disk.")
    if target:
        result = download_many(target, api_key, P["nl_root"],
                               log_path=log_path, max_workers=8)
        missing = result.get("unavailable", 0) + result.get("failed", 0)
        print("All files downloaded." if missing == 0
              else f"{missing} file(s) not downloaded -- see {log_path}")
    else:
        print("Nothing missing: up to date.")
else:
    print("No token -> skipped. Working with what is already on disk.")

rasterFiles = list_tile_rasters(P["nl_root"], list(cfg.TILES), make=False)
print(f"\n{len(rasterFiles):,} raster file(s) available across {cfg.TILES}")

# Measuring a country

## Step 1 — Boundary

We take the country outline from **GADM**, the standard open administrative-boundary
database. `cfg.GADM_ZIP` is built from the ISO3 code in the registry.

In [ ]:
import geopandas as gpd

shape_url   = "https://geodata.ucdavis.edu/gadm/gadm4.1/shp/" + cfg.GADM_ZIP
print("Boundary:", shape_url)

gdf         = gpd.read_file(shape_url)
gdf_country = gdf[gdf.geometry.notnull()].dissolve().to_crs("EPSG:4326")

print(f"{cfg.COUNTRY}: {len(gdf)} administrative unit(s) dissolved into one polygon")
print(f"bounds: {tuple(round(b, 2) for b in gdf_country.total_bounds)}")

gdf_country.explore(tiles="CartoDB dark_matter")

## Step 2 — two questions, two functions

| Question | Function | What you give it |
|---|---|---|
| How lit is this **area**? | `processHD5_shape` | a boundary polygon |
| How lit is this **site**? | `processHD5` | a list of lat/lon points |

Both in `ntl_helpers.py`.

In [ ]:
import numpy as np
from tqdm import tqdm

if cfg.NTL_DATE_RANGE:
    _ws, _we = (pd.to_datetime(x) for x in cfg.NTL_DATE_RANGE.split())
else:
    _ws = _we = None

files = rasters_in_range(P["nl_root"], list(cfg.TILES), start=_ws, end=_we)
print(f"{len(files)} raster(s) in the window\n")

recs = []
for _path, _dt in tqdm(files, desc="Country NTL"):
    try:
        recs.append(processHD5_shape(_path, 2, P["ntl_tmp"], gdf_country, _dt))
    except Exception as e:
        print(f"  skip {os.path.basename(_path)}: {type(e).__name__}: {e}")

shape_df = pd.DataFrame(recs)
shape_df["date"] = pd.to_datetime(shape_df["date"])

# One row per date. Across tiles: totals add, means average, pixel counts add.
shape_df = (shape_df.groupby("date")
                    .agg(ntl_mean=("ntl_mean", "mean"),
                         ntl_sum=("ntl_sum", "sum"),
                         ntl_n=("ntl_n", "sum"))
                    .sort_index())

shape_df.to_csv(os.path.join(P["raw"], cfg.NTL_SHAPE_CSV))
print(f"\nSaved {len(shape_df)} rows -> raw/{cfg.NTL_SHAPE_CSV}")
shape_df.tail()

## Step 3 — the daily series

In [ ]:
import matplotlib.pyplot as plt

win_start, win_end = shape_df.index.min(), shape_df.index.max()

if cfg.NTL_EVENT_DATE:
    marks = [(pd.Timestamp(cfg.NTL_EVENT_DATE), cfg.NTL_EVENT_LABEL or "event")]
else:
    marks = [(win_start, "start"), (win_end, "end")]

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(shape_df.index, shape_df["ntl_mean"], color="#005BAC", linewidth=1.8,
        marker="o", markersize=3, label="NTL mean (nW/cm²/sr)")
for _d, _lab in marks:
    ax.axvline(_d, color="#CC0000", linewidth=1.5, linestyle="--", zorder=3)
    ax.text(_d, ax.get_ylim()[1], f" {_lab}", color="#CC0000", fontsize=9,
            va="top", ha="left")

ax.set_title(f"{cfg.COUNTRY} — nighttime lights, country mean",
             fontsize=13, weight="bold")
ax.set_ylabel("nW/cm²/sr")
ax.grid(axis="y", linestyle=":", alpha=0.5)
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## Step 4 — before and after, on a map

The same two dates, drawn as rasters. With an event set you get `event ±
NTL_EVENT_WINDOW` days; without one, the first and last date of the window.

In [ ]:
import contextily as cx
import colorcet as cc

if cfg.NTL_EVENT_DATE:
    _ev = pd.Timestamp(cfg.NTL_EVENT_DATE)
    d_before = _ev - pd.Timedelta(days=cfg.NTL_EVENT_WINDOW)
    d_after  = _ev + pd.Timedelta(days=cfg.NTL_EVENT_WINDOW)
    _lab = cfg.NTL_EVENT_LABEL or "event"
    lab_before = f"{cfg.NTL_EVENT_WINDOW} days before {_lab}"
    lab_after  = f"{cfg.NTL_EVENT_WINDOW} days after {_lab}"
else:
    d_before, d_after = shape_df.index.min(), shape_df.index.max()
    lab_before, lab_after = "window start", "window end"

# ALL tiles for each date, not just the first. A country that straddles a tile
# boundary (The Bahamas = h09v06 + h10v06) is drawn from the mosaic of both; take
# only the first and you map whichever 10-degree square sorts first, which here is
# the strip of open ocean west of longitude -80.
f_before = find_h5s_for_date(d_before, P["nl_root"], list(cfg.TILES))
f_after  = find_h5s_for_date(d_after,  P["nl_root"], list(cfg.TILES))
_tiles_of = lambda fs: " + ".join(os.path.basename(f).split(".")[2] for f in fs) or "MISSING"
print(f"before ({d_before.date()}): {len(f_before)}/{len(cfg.TILES)} tile(s)  {_tiles_of(f_before)}")
print(f"after  ({d_after.date()}): {len(f_after)}/{len(cfg.TILES)} tile(s)  {_tiles_of(f_after)}")

if not f_before or not f_after:
    print("\nNo raster for one of those dates -- skipping the maps.")
else:
    arr_b, lons, lats = raster_to_array(f_before, 2, gdf_country, P["ntl_tmp"])
    arr_a, _, _       = raster_to_array(f_after,  2, gdf_country, P["ntl_tmp"])

    both = np.concatenate([arr_b[~np.isnan(arr_b)], arr_a[~np.isnan(arr_a)]])
    vmin, vmax = np.nanpercentile(both, 2), np.nanpercentile(both, 98)
    LON, LAT = np.meshgrid(lons, lats)

    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    fig.subplots_adjust(bottom=0.15)
    titles = [f"{lab_before}\n({d_before.strftime('%b %d, %Y')})",
              f"{lab_after}\n({d_after.strftime('%b %d, %Y')})"]
    for ax, arr, title in zip(axes, [arr_b, arr_a], titles):
        pcm = ax.pcolormesh(LON, LAT, arr, cmap=cc.cm.bmy,
                            vmin=vmin, vmax=vmax, shading="auto")
        ax.set_title(title, fontsize=13, weight="bold")
        try:
            cx.add_basemap(ax, source=cx.providers.CartoDB.Positron, crs="EPSG:4326")
        except Exception as e:
            print(f"  basemap unavailable ({type(e).__name__}) -- plotting without it")
        ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")

    cbar_ax = fig.add_axes([0.25, 0.05, 0.5, 0.025])
    fig.colorbar(pcm, cax=cbar_ax, orientation="horizontal",
                 label="NTL radiance (nW/cm²/sr)")
    fig.suptitle(f"Nighttime light radiance — {cfg.COUNTRY}", fontsize=15, weight="bold")
    fig.text(0.01, 0.01, "Source: NASA Black Marble VNP46A2", fontsize=10)
    plt.show()

## Step 5 — individual sites

The country mean is a blunt instrument. A hotel strip, a port, an airport, a refinery
each have their own series, and those are what connect to the sectors you are trying
to nowcast — tourism, trade, energy.

`raw/Coordinates.xlsx` holds the site list for each country: sector, name, city,
latitude, longitude. `processHD5` returns two readings per site per night:

* `DNBvalue1` — the single pixel containing the point
* `DNBvalue3` — the mean of the 3×3 pixel block around it

Prefer `DNBvalue3` in practice. A VIIRS pixel is about 464 m across and geolocation is
not perfect, so a single pixel can miss a site by one cell; the 3×3 mean is far more
stable. The flip side is that sites closer together than ~464 m share pixels and are
not really independent series.

In [ ]:
coord_path = os.path.join(P["raw"], cfg.COORDINATES_XLSX)

if not os.path.exists(coord_path):
    print(f"No {cfg.COORDINATES_XLSX} in raw/ -- skipping the site-level extraction.")
    site_df = pd.DataFrame()
else:
    locations = pd.read_excel(coord_path)

    # Strip accents and stray whitespace so names stay stable as merge keys.
    for col in ("location", "city", "sector"):
        locations[col] = (locations[col].fillna("").astype(str)
                            .str.normalize("NFKD")
                            .str.encode("ascii", errors="ignore").str.decode("utf-8")
                            .str.strip())

    coords = locations[["sector", "location", "city", "lat", "lon"]].to_dict("records")
    print(f"{len(coords)} site(s) in {cfg.COORDINATES_XLSX}")
    print("sectors:", ", ".join(sorted(locations['sector'].unique())), "\n")

    rows = []
    for _path, _dt in tqdm(files, desc="Site NTL"):
        try:
            rows.extend(processHD5(_path, 2, P["ntl_tmp"], coords, _dt))
        except Exception as e:
            print(f"  skip {os.path.basename(_path)}: {type(e).__name__}: {e}")

    site_df = pd.DataFrame(rows).rename(columns={"JD": "date"})
    site_df["date"] = pd.to_datetime(site_df["date"])
    site_df = site_df.sort_values(["location", "date"])

    site_df.to_csv(os.path.join(P["raw"], cfg.NTL_OUTPUT_CSV), index=False)
    print(f"\nSaved {len(site_df):,} site-days -> raw/{cfg.NTL_OUTPUT_CSV}")

site_df.head()

In [ ]:
# Mean brightness by sector over the window -- the site data in one picture.
if len(site_df):
    by_sector = (site_df.groupby(["date", "sector"])["DNBvalue3"]
                        .mean().unstack("sector"))

    ax = by_sector.plot(figsize=(12, 4), linewidth=1.6, marker="o", markersize=3)
    ax.set_title(f"{cfg.COUNTRY} — mean site radiance by sector", fontsize=13, weight="bold")
    ax.set_ylabel("nW/cm²/sr")
    ax.set_xlabel("")
    ax.grid(axis="y", linestyle=":", alpha=0.5)
    ax.legend(frameon=False, ncol=3, fontsize=9)
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.show()

    display(by_sector.describe().T[["count", "mean", "std", "min", "max"]].round(2))